Dataset Creation:

In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Generazione date (ultimi 30 giorni)

dates = pd.date_range(end=datetime.today(), periods=24*30, freq='h')

# Dataset Produzione Solare (con qualche dato sporco/nullo)

production = np.maximum(0, np.sin((dates.hour.to_numpy() - 6) * np.pi / 12) * 5 + np.random.normal(0, 0.5, len(dates)))

production[dates.hour < 6] = 0 # Notte

production[dates.hour > 20] = 0 # Notte

df_prod = pd.DataFrame({'timestamp': dates, 'kwh_produced': production})


# Introduciamo anomalie (dati sporchi)

df_prod.loc[100:105, 'kwh_produced'] = np.nan # Sensore rotto

df_prod.loc[200, 'kwh_produced'] = -5 # Errore sensore (valore negativo impossibile)


# Dataset Consumi Utente

consumption = np.random.normal(1.5, 0.5, len(dates)) # Consumo base

consumption[dates.hour.isin([19, 20, 21])] += 2 # Picco serale

df_cons = pd.DataFrame({'timestamp': dates, 'kwh_consumed': consumption})


# Salvataggio

df_prod.to_csv("solar_production_raw.csv", index=False)

df_cons.to_csv("household_consumption.csv", index=False)

print("Dataset generati: 'solar_production_raw.csv' e 'household_consumption.csv'")

Dataset generati: 'solar_production_raw.csv' e 'household_consumption.csv'


Cell 2 — Library Verification

In [ ]:
import langchain
import streamlit

# Prints installed versions to verify the environment is correctly set up
print(f"LangChain: {langchain.__version__}")
print(f"Streamlit: {streamlit.__version__}")

LangChain: 1.2.15
Streamlit: 1.56.0


## Imports & LLM Initialization

Imports all dependencies and configures the language model.

**AI Stack:**
- `ChatGroq` — client for `llama-3.3-70b-versatile` via the Groq API
- `create_react_agent` (LangGraph) — builds the agent using the **ReAct** pattern (Reason + Act)
- API key is loaded from a `.env` file via `python-dotenv`
- `@lru_cache(maxsize=1)` stores the result in memory after the first read —
  subsequent tool calls skip disk I/O entirely

In [ ]:
#Import libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from functools import lru_cache

#Langchain
from langchain_groq import ChatGroq   # Groq LLM provider
from langchain_core.tools import tool  
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent # ReAct agent builder. Reasoning + Acting: the model reasons, decides which tool to call, observes the result, and iterates until it reaches a final answer.

load_dotenv() #Load GROQ_API_KEY from .env file
api_key = os.getenv('GROQ_API_KEY')

# Initialize the LLM
llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature= 0,      # 0 = no randomness, fully reproducible responses
    api_key = api_key
)

 `load_data()`

Utility function shared by all tools.

- Tries to load **cleaned** CSVs first (`solar_production_clean.csv`, `household_consumption_clean.csv`)
- Falls back to **raw** files if clean versions don't exist yet (first run)
- Parses `timestamp` column to `datetime` once here, avoiding repetition in every tool
- `@lru_cache(maxsize=1)` stores the result in memory after the first read —
  subsequent tool calls skip disk I/O entirely

In [ ]:
@lru_cache(maxsize=1)

def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Loads cleaned CSVs, falls back to raw if not found.
    Reads CSVs once, then serves from memory on every subsequent tool call."""
    try:
        df_prod = pd.read_csv('solar_production_clean.csv')
        df_cons = pd.read_csv('household_consumption_clean.csv')
     # Falls back to raw data if clean version doesn't exist yet
    except FileNotFoundError:
        df_prod = pd.read_csv('solar_production_raw.csv')
        df_cons = pd.read_csv('household_consumption.csv')
    
    # Converti timestamp una volta sola
    df_prod['timestamp'] = pd.to_datetime(df_prod['timestamp'])
    df_cons['timestamp'] = pd.to_datetime(df_cons['timestamp'])
    return df_prod, df_cons

Cell 5 — clean_data Tool

Must always run first. Raw files contain NaN gaps and negative values that silently corrupt all downstream statistics if not fixed.

In [ ]:
@tool
def clean_data()-> str:
    '''
    Loads and cleans raw IoT sensor CSVs.
    - Interpolates NaN values in solar production
    - Replaces negative values with 0 (physically impossible)
    - Saves cleaned versions as solar_production_clean.csv and household_consumption_clean.csv
    - Must always be called FIRST before any analysis tool
    '''
    #Import csv files
    try:
        df_production = pd.read_csv('solar_production_raw.csv', parse_dates=["timestamp"])
        df_consumption = pd.read_csv("household_consumption.csv", parse_dates=["timestamp"])

        report = []

        # Fixing NaN in produtcion with linear interpolation
        #Linear interpolation: estimates NaNs based on neighboring values. Appropriate for continuous time series like solar production.
        null_values = df_production['kwh_produced'].isnull().sum()
        df_production['kwh_produced'] = df_production['kwh_produced'].interpolate(method='linear')
        report.append(f"{null_values} NaN values → interpolated")


        # Fix negatives in production assigning 0
        negative_values = (df_production['kwh_produced']<0).sum()
        df_production.loc[df_production['kwh_produced']<0 ,'kwh_produced']=0
        report.append(f"{negative_values} negatives values → sobstituted with 0")


        # Fix negatives in consumption
        negative_values_con =  (df_consumption['kwh_consumed']<0).sum()
        df_consumption.loc[df_consumption['kwh_consumed']<0, 'kwh_consumed']=0
        report.append(f"{negative_values_con} consumption negatives values → sobstituted with 0")


        #NaN in consuption are already equals to  0


        # Persist cleaned data
        df_production.to_csv('solar_production_clean.csv', index=False)
        df_consumption.to_csv('household_consumption_clean.csv', index=False)
        
        load_data.cache_clear()
        return "Cleaning completed\n" + "\n".join(report)
    
    except Exception as e:
        return f"Errore: {str(e)}"

Cell 6 — analyze_production Tool

Computes descriptive statistics on solar production:
- Total kWh produced over the period
- Hourly average
- Best and worst production day
- Peak production hour


In [ ]:
@tool

def analyze_production () -> str:
    '''
    Reads the cleaned solar production CSV and returns descriptive statistics:
    total kWh produced, hourly average, best/worst day, and peak production hour.
    Use when user asks about solar panels or energy production.
    '''

    try:
        df, _ = load_data()
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour

        total = df['kwh_produced'].sum()
        avarage = df['kwh_produced'].mean()

        daily_total = df.groupby('date')['kwh_produced'].sum()
        max_daily = daily_total.idxmax()
        min_daily = daily_total.idxmin()

        per_hour = df.groupby('hour')['kwh_produced'].mean()
        maximum_daily_peak = per_hour.idxmax()

        Analysis = f"""
        Total energy produced: {total:.2f} kWh
        Average per hour: {avarage:.2f} kWh
        Best production day: {max_daily}
        Worst production day: {min_daily}
        Peak production hour: {maximum_daily_peak}:00
        Days analyzed: {len(daily_total)}
        """

        return Analysis.strip()

    except Exception as e:
        return f"Errore: {str(e)}"

Cell 7 — user_consumption Tool
Descriptive statistics on household energy consumption:
- Total kWh consumed
- Hourly average
- Days with peak maximum/minimum consumption
- Peak consumption hour


In [ ]:
@tool
def user_consumption()->str:
    '''
   Reads cleaned consumption CSV and returns descriptive statistics:
    total consumed, hourly average, peak consumption day/hour.
    Use when user asks about household consumption, bills, or energy habits.
    '''

    # Same fallback pattern as analyze_production
    try:
        _, df = load_data()

    
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour

        total_con = df['kwh_consumed'].sum()
        avarage_con = df['kwh_consumed'].mean()

        daily_total_con = df.groupby('date')['kwh_consumed'].sum()
        maximum_daily_peak = daily_total_con.idxmax()
        mininum_daily_peak = daily_total_con.idxmin()

        per_hour_con = df.groupby('hour')['kwh_consumed'].mean()
        maximum_hourly_peak = per_hour_con.idxmax()


        Analysis = f'''
        Total energy consumed: {total_con:.2f} kWh
        Average per hour: {avarage_con:.2f} kWh
        Maximum energy consumed in a day: {maximum_daily_peak} kWh
        Minimum energy consumed in a day: {mininum_daily_peak} kWh
        Peak consumption hour: {maximum_hourly_peak}:00
        Number of days analyzed: {len(daily_total_con)}
        '''
        return Analysis.strip()
    
    except Exception as e:
        return f"Errore: {str(e)}"

Cell 8 — charts_design Tool
Generates a PNG with two stacked charts saved to `charts/analisi_energetica.png`:

1. **Daily production** — line chart with dates on the x-axis
2. **Production vs Consumption** — overlapping comparison to visually identify surplus and deficits

The chart is displayed in the Streamlit UI only when the user's message contains trigger words
like `"chart"`, `"graph"`, `"grafico"`, etc.

In [ ]:
@tool

def charts_design()->str:
    '''
    Generates two matplotlib charts saved as charts/analisi_energetica.png:
    1. Daily solar production line chart (with date labels on x-axis)
    2. Overlay comparison: production vs consumption (indexed by day number)
    Returns the file path of the saved PNG.
    Use when user asks for a chart, graph, or visual representation.
    '''
    try:
        df_production, df_consumption = load_data() 

        os.makedirs('charts', exist_ok=True)

        df_consumption['date'] = df_consumption['timestamp'].dt.date
        df_production['date'] = df_production['timestamp'].dt.date

        daily_consume = df_consumption.groupby('date')['kwh_consumed'].sum()
        daily_production = df_production.groupby('date')['kwh_produced'].sum()

        # charts

        fig, axes = plt.subplots(2, 1, figsize=(12,8))

        # Chart 1: production over time
        axes[0].plot(daily_production.index, daily_production.values, linewidth=2, marker="o", markersize=3)
        axes[0].set_title('Daily energy production')
        axes[0].set_ylabel('Kwh')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(True, alpha=0.3)


        # Chart 2: production vs consumption
        axes[1].plot(daily_production.index, daily_production.values, label='Production', linewidth=2)
        axes[1].plot(daily_consume.index, daily_consume.values, label='Consumption', linewidth=2)
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].set_title('Production vs Consumption')
        axes[1].set_ylabel('Kwh')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        #Save png
        filepath = "charts/analisi_energetica.png"
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close()

        return f"Chart saved in: {filepath}"

    except Exception as e:
        return f"Chart Generation Error: {str(e)}"

Cell 9 — advisor Tool

Generates personalized recommendations to optimize energy usage:

- **Surplus hours** — identifies when production exceeds consumption (optimal windows for running
  the washing machine, dishwasher, charging devices, etc.)
- **Evening peak** — warns if maximum consumption falls after sunset, when panels produce nothing
- **Energy self-sufficiency** — computes `min(100, production/consumption * 100)%`:
  - ≥ 50%: efficient use of renewables
  - < 50%: suggests investing in a storage battery

In [ ]:
@tool
def advisor()->str:
    """
    Compares hourly production vs consumption to generate personalized energy-saving advice:
    - Identifies surplus hours (best time to run appliances)
    - Flags evening peak vs production mismatch
    - Calculates energy self-sufficiency % and suggests battery storage if < 50%
    Use when user asks for tips, savings advice, or optimization suggestions.
    """
    try:
        df_production, df_consumption = load_data()

        df_consumption['hour'] = df_consumption['timestamp'].dt.hour
        df_production['hour'] = df_production['timestamp'].dt.hour

        hourly_consumption = df_consumption.groupby('hour')['kwh_consumed'].mean()
        hourly_production = df_production.groupby('hour')['kwh_produced'].mean()

        # Surplus is computed per hour of day using average profiles.
        # A positive surplus at hour means that, on average, the panels produce more than the household consumes at that time — making it the optimal window to schedule high-demand appliances.

        #surplus elettrico
        surplus =  hourly_production - hourly_consumption
        ore_surplus =  surplus[surplus>0].index.tolist()

        peak_hour_cons = hourly_consumption.idxmax()

        total_production = df_production['kwh_produced'].sum()
        total_consumption =    df_consumption['kwh_consumed'].sum()


        # Self-sufficiency measures what fraction of total consumption is covered by local solar production. 
        # Values below 50% suggest that more than half the energy is drawn from the grid, making battery storage a financially viable option to consider
        Self_sufficiency = min(100,(total_production/total_consumption)*100)

        Advices = []

        if ore_surplus:
            ore_s = ', '.join([f'{o}:00' for o in ore_surplus[:5]])
            Advices.append(
                f'During the hours {ore_s} you produce more than you consume. '
                f'It is the ideal time to run the washing machine, dishwasher, or charge your devices.'
                )


        if peak_hour_cons > 18:
            Advices.append(
                f'Your consumption peak is at {peak_hour_cons}:00 (evening), '
                f'when the solar panels are not producing. Try to shift some loads '
                f'to the central hours of the day.'
                )

        
        if Self_sufficiency <50:
            Advices.append(
                f'Your energy self-sufficiency is {Self_sufficiency:.1f}%. '
                f'You could consider purchasing a storage battery to '
                f'store the energy produced during the day and use it in the evening.'
            )
        else:
            Advices.append(
                f'Great! Your energy self-sufficiency is {Self_sufficiency:.1f}%. '
                f'You are making excellent use of your community\'s renewable energy.'
            )
        return "\n".join(Advices)
        
    except Exception as e:
        return f'Errore: {str(e)}'

-cell 10
 `forecast_production(lat, lon, kwp)`

Calls the **Forecast.Solar** free API to estimate today's hourly solar production.

**Parameters** (all passed as strings by the LLM):
- `lat`, `lon` — coordinates (default: `-12.04`, `-77.03` → Lima, Peru)
- `kwp` — installed peak power in kilowatts (default: `5.0`)

In [ ]:
import requests

@tool
def forecast_production(lat: str, lon: str, kwp: str) -> str:
    """
    Estimates solar production for today using Forecast.Solar API.
    Use when user asks about future or expected solar production.
    Args:Use these default values unless the user specifies otherwise:
    lat="-12.04", lon="-77.03", kwp="5.0"
    """
    try:
        # Args arrive as strings from the LLM — cast to float for the URL
        lat, lon, kwp = float(lat), float(lon), float(kwp)
        url = f"https://api.forecast.solar/estimate/{lat}/{lon}/30/0/{kwp}"
        data = requests.get(url, timeout=10).json()

        # watt_hours_period: energy produced in each 1-hour slot (in Wh)
        wh_per_hour = data['result']['watt_hours_period']

        # Convert each slot from Wh → kWh for readability
        lines = [f"  {hour} → {wh/1000:.2f} kWh"
                 for hour, wh in wh_per_hour.items()]
        
        # Total daily production in kWh
        total = sum(wh_per_hour.values()) / 1000

        return f"☀️ Forecast today:\n" + "\n".join(lines) + \
               f"\n\n⚡ Total estimated: {total:.2f} kWh"

    except Exception as e:
        return f"❌ Forecast error: {str(e)}"

Cell 11 — Agent Creation

In [ ]:
# Registers all 5 tools and binds them to the LLM as a ReAct agent
tools = [
    clean_data,
    analyze_production,
    user_consumption,
    charts_design,
    advisor,
    forecast_production
]

systemprompt = """
You are EcoBot, a virtual energy analyst for EcoGrid Solutions.
You help families understand energy data in a clear, accurate, and safe way.

CORE RULES
- Always use the available tools as the only source of truth for numerical answers.
- Never invent, estimate, infer, or assume values that are not explicitly returned by a tool.
- Never say "I can't", "I'm unable", or "I cannot provide" — always attempt to use the appropriate tool first.
- Always respond in the same language as the user.
- Never answer a numerical question from memory or from previous conversation alone.

MANDATORY WORKFLOW
- For the first data question, call clean_data first.
- Then call the most relevant tool for the user's request.
- Questions about consumption statistics → user_consumption
- Questions about production statistics → analyze_production
- Questions about charts or visualizations → charts_design
- Questions about advice or savings suggestions → advisor
- Questions about future/forecast/expected production → forecast_production
  Use lat="-12.04", lon="-77.03", kwp="5.0" as defaults unless the user specifies otherwise.

SAFETY BEHAVIOR
- If the tool output is insufficient to answer exactly, say so clearly.
- Do not "enrich" missing data.
- Only explain, format, or summarize what has been verified by the tool.
- If a tool fails, report the error from the tool, not a generic refusal.

RESPONSE STYLE
- Be concise, clear, and practical.
- Quote the exact values returned by the tool when useful.
- Avoid contradictions with previous answers.
"""

agent = create_react_agent(
    model = llm,
    tools= tools,
    prompt=systemprompt
)

print(f"Tools disponibili: {[t.name for t in tools]}")


Tools disponibili: ['clean_data', 'analyze_production', 'user_consumption', 'charts_design', 'advisor', 'forecast_production']


C:\Users\Usuario\AppData\Local\Temp\ipykernel_1316\2810306335.py:44: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Cell 12 — Agent Test Helper


In [ ]:
def chiedi_a_ecobot(question: str):
    """ Utility function to test the agent in the notebook (outside Streamlit).
    Sends a HumanMessage and prints the last response message."""
    print(f"\n User: {question}")
    print("─" * 50)
    
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })
    
    # The AI message is always in the last message of the list
    answer = result["messages"][-1].content
    print(f"EcoBot: {answer}")
    return answer

STREAMLIT
Writes the entire Streamlit app to disk as app.py
Key components:
- Sidebar: logo, CSV file uploaders (production + consumption)
- Session state: persists agent instance and chat history across reruns
- On file upload: saves CSVs to disk, creates the agent via agente()
- Chat interface: user input → agent.invoke() → display response
- Auto-displays chart PNG if user asks for a "chart" and the file exists

In [ ]:
%%writefile app.py

import streamlit as st       
import pandas as pd
import os
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from langchain_groq import ChatGroq 
from langchain_core.tools import tool       
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
import requests

load_dotenv()
st.set_page_config(
    page_title="EcoBot — EcoGrid Solutions",
    page_icon="⚡",
    layout="wide")

# --- Session State Initialization ---
# Ensure all required session variables are defined before use.
# This avoids runtime errors and preserves state across reruns.
if 'messages' not in st.session_state:
    st.session_state.messages = []

if 'agent' not in st.session_state:
    st.session_state.agent = None

if 'uploaded_data' not in st.session_state:
    st.session_state.uploaded_data = False

#import tools used before
#------------------------------------------------------------------------------
@tool
def clean_data()-> str:
    '''
    Loads and cleans raw IoT sensor CSVs.
    - Interpolates NaN values in solar production
    - Replaces negative values with 0 (physically impossible)
    - Saves cleaned versions as solar_production_clean.csv and household_consumption_clean.csv
    - Must always be called FIRST before any analysis tool
    '''
    try:
        df_production = pd.read_csv('solar_production_raw.csv', parse_dates=["timestamp"])
        df_consumption = pd.read_csv("household_consumption.csv", parse_dates=["timestamp"])

        report = []

        # Fixing NaN in produtcion with linear interpolation
        #Linear interpolation: estimates NaNs based on neighboring values. Appropriate for continuous time series like solar production.
        null_values = df_production['kwh_produced'].isnull().sum()
        df_production['kwh_produced'] = df_production['kwh_produced'].interpolate(method='linear')
        report.append(f"{null_values} NaN values → interpolated")


        # Fix negatives in production assigning 0
        negative_values = (df_production['kwh_produced']<0).sum()
        df_production.loc[df_production['kwh_produced']<0 ,'kwh_produced']=0
        report.append(f"{negative_values} negatives values → sobstituted with 0")


        # Fix negatives in consumption
        negative_values_con =  (df_consumption['kwh_consumed']<0).sum()
        df_consumption.loc[df_consumption['kwh_consumed']<0, 'kwh_consumed']=0
        report.append(f"{negative_values_con} consumption negatives values → sobstituted with 0")


        #NaN in consuption are already equals to  0


        # Persist cleaned data
        df_production.to_csv('solar_production_clean.csv', index=False)
        df_consumption.to_csv('household_consumption_clean.csv', index=False)

        return "Cleaning completed\n" + "\n".join(report)
    
    except Exception as e:
        return f"Errore: {str(e)}"

#------------------------------------------------------------------------------
@tool

def analyze_production () -> str:
    '''
    Reads the cleaned solar production CSV and returns descriptive statistics:
    total kWh produced, hourly average, best/worst day, and peak production hour.
    Use when user asks about solar panels or energy production.
    '''

    try:
        df, _ = load_data()
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour

        total = df['kwh_produced'].sum()
        avarage = df['kwh_produced'].mean()

        daily_total = df.groupby('date')['kwh_produced'].sum()
        max_daily = daily_total.idxmax()
        min_daily = daily_total.idxmin()

        per_hour = df.groupby('hour')['kwh_produced'].mean()
        maximum_daily_peak = per_hour.idxmax()

        Analysis = f"""
        Total energy produced: {total:.2f} kWh
        Average per hour: {avarage:.2f} kWh
        Best production day: {max_daily}
        Worst production day: {min_daily}
        Peak production hour: {maximum_daily_peak}:00
        Days analyzed: {len(daily_total)}
        """

        return Analysis.strip()

    except Exception as e:
        return f"Errore: {str(e)}"

#------------------------------------------------------------------------------
@tool
def user_consumption()->str:
    '''
   Reads cleaned consumption CSV and returns descriptive statistics:
    total consumed, hourly average, peak consumption day/hour.
    Use when user asks about household consumption, bills, or energy habits.
    '''

    # Same fallback pattern as analyze_production
    try:
        _, df = load_data()

    
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour

        total_con = df['kwh_consumed'].sum()
        avarage_con = df['kwh_consumed'].mean()

        daily_total_con = df.groupby('date')['kwh_consumed'].sum()
        maximum_daily_peak = daily_total_con.idxmax()
        mininum_daily_peak = daily_total_con.idxmin()

        per_hour_con = df.groupby('hour')['kwh_consumed'].mean()
        maximum_hourly_peak = per_hour_con.idxmax()


        Analysis = f'''
        Total energy consumed: {total_con:.2f} kWh
        Average per hour: {avarage_con:.2f} kWh
        Maximum energy consumed in a day: {maximum_daily_peak} kWh
        Minimum energy consumed in a day: {mininum_daily_peak} kWh
        Peak consumption hour: {maximum_hourly_peak}:00
        Number of days analyzed: {len(daily_total_con)}
        '''
        return Analysis.strip()
    
    except Exception as e:
        return f"Errore: {str(e)}"


#------------------------------------------------------------------------------
@tool

def charts_design()->str:
    '''
    Generates two matplotlib charts saved as charts/analisi_energetica.png:
    1. Daily solar production line chart (with date labels on x-axis)
    2. Overlay comparison: production vs consumption (indexed by day number)
    Returns the file path of the saved PNG.
    Use when user asks for a chart, graph, or visual representation.
    '''
    try:
        df_production, df_consumption = load_data() 

        os.makedirs('charts', exist_ok=True)

        df_consumption['date'] = df_consumption['timestamp'].dt.date
        df_production['date'] = df_production['timestamp'].dt.date

        daily_consume = df_consumption.groupby('date')['kwh_consumed'].sum()
        daily_production = df_production.groupby('date')['kwh_produced'].sum()

        # charts

        fig, axes = plt.subplots(2, 1, figsize=(12,8))

        # Chart 1: production over time
        axes[0].plot(daily_production.index, daily_production.values, linewidth=2, marker="o", markersize=3)
        axes[0].set_title('Daily energy production')
        axes[0].set_ylabel('Kwh')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(True, alpha=0.3)


        # Chart 2: production vs consumption
        axes[1].plot(daily_production.index, daily_production.values, label='Production', linewidth=2)
        axes[1].plot(daily_consume.index, daily_consume.values, label='Consumption', linewidth=2)
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].set_title('Production vs Consumption')
        axes[1].set_ylabel('Kwh')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        #Save png
        filepath = "charts/analisi_energetica.png"
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close()

        return f"Chart saved in: {filepath}"

    except Exception as e:
        return f"Chart Generation Error: {str(e)}"
    
    #------------------------------------------------------------------------------
@tool
def advisor()->str:
    """
    Compares hourly production vs consumption to generate personalized energy-saving advice:
    - Identifies surplus hours (best time to run appliances)
    - Flags evening peak vs production mismatch
    - Calculates energy self-sufficiency % and suggests battery storage if < 50%
    Use when user asks for tips, savings advice, or optimization suggestions.
    """
    try:
        df_production, df_consumption = load_data()

        df_consumption['hour'] = df_consumption['timestamp'].dt.hour
        df_production['hour'] = df_production['timestamp'].dt.hour

        hourly_consumption = df_consumption.groupby('hour')['kwh_consumed'].mean()
        hourly_production = df_production.groupby('hour')['kwh_produced'].mean()

        # Surplus is computed per hour of day using average profiles.
        # A positive surplus at hour H means that, on average, the panels produce more than the household consumes at that time — making it the optimal window to schedule high-demand appliances.
        #surplus elettrico
        surplus =  hourly_production - hourly_consumption
        ore_surplus =  surplus[surplus>0].index.tolist()

        peak_hour_cons = hourly_consumption.idxmax()

        total_production = df_production['kwh_produced'].sum()
        total_consumption =    df_consumption['kwh_consumed'].sum()


        # Self-sufficiency measures what fraction of total consumption is covered by local solar production. 
        # Values below 50% suggestthat more than half the energy is drawn from the grid, making battery storage a financially viable option to consider
        Self_sufficiency = min(100,(total_production/total_consumption)*100)

        Advices = []

        if ore_surplus:
            ore_s = ', '.join([f'{o}:00' for o in ore_surplus[:5]])
            Advices.append(
                f'During the hours {ore_s} you produce more than you consume. '
                f'It is the ideal time to run the washing machine, dishwasher, or charge your devices.'
                )


        if peak_hour_cons > 18:
            Advices.append(
                f'Your consumption peak is at {peak_hour_cons}:00 (evening), '
                f'when the solar panels are not producing. Try to shift some loads '
                f'to the central hours of the day.'
                )

        
        if Self_sufficiency <50:
            Advices.append(
                f'Your energy self-sufficiency is {Self_sufficiency:.1f}%. '
                f'You could consider purchasing a storage battery to '
                f'store the energy produced during the day and use it in the evening.'
            )
        else:
            Advices.append(
                f'Great! Your energy self-sufficiency is {Self_sufficiency:.1f}%. '
                f'You are making excellent use of your community\'s renewable energy.'
            )
        return "\n".join(Advices)
        
    except Exception as e:
        return f'Errore: {str(e)}'

#---------------------------------------------------------------------------------------
@tool
def forecast_production(lat: str, lon: str, kwp: str) -> str:
    """
    Estimates solar production for today using Forecast.Solar API.
    Use when user asks about future or expected solar production.
    Args:Use these default values unless the user specifies otherwise:
    lat="-12.04", lon="-77.03", kwp="5.0"
    """
    try:
        lat, lon, kwp = float(lat), float(lon), float(kwp)
        url = f"https://api.forecast.solar/estimate/{lat}/{lon}/30/0/{kwp}"
        data = requests.get(url, timeout=10).json()

        wh_per_hour = data['result']['watt_hours_period']

        lines = [f"  {hour} → {wh/1000:.2f} kWh"
                 for hour, wh in wh_per_hour.items()]

        total = sum(wh_per_hour.values()) / 1000

        return f"☀️ Forecast today:\n" + "\n".join(lines) + \
               f"\n\n⚡ Total estimated: {total:.2f} kWh"

    except Exception as e:
        return f"❌ Forecast error: {str(e)}"

#------------------------------ Agent creation ------------------------------------------

def agente():
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY")
        )

    tools = [clean_data, analyze_production, user_consumption, charts_design, advisor, forecast_production]

    systemprompt = """
You are EcoBot, a virtual energy analyst for EcoGrid Solutions.
You help families understand energy data in a clear, accurate, and safe way.

CORE RULES
- Always use the available tools as the only source of truth for numerical answers.
- Never invent, estimate, infer, or assume values that are not explicitly returned by a tool.
- Never say "I can't", "I'm unable", or "I cannot provide" — always attempt to use the appropriate tool first.
- Always respond in the same language as the user.
- Never answer a numerical question from memory or from previous conversation alone.

MANDATORY WORKFLOW
- For the first data question, call clean_data first.
- Then call the most relevant tool for the user's request.
- Questions about consumption statistics → user_consumption
- Questions about production statistics → analyze_production
- Questions about charts or visualizations → charts_design
- Questions about advice or savings suggestions → advisor
- Questions about future/forecast/expected production → forecast_production
  Use lat="-12.04", lon="-77.03", kwp="5.0" as defaults unless the user specifies otherwise.

SAFETY BEHAVIOR
- If the tool output is insufficient to answer exactly, say so clearly.
- Do not "enrich" missing data.
- Only explain, format, or summarize what has been verified by the tool.
- If a tool fails, report the error from the tool, not a generic refusal.

RESPONSE STYLE
- Be concise, clear, and practical.
- Quote the exact values returned by the tool when useful.
- Avoid contradictions with previous answers.
"""

    return create_react_agent(model=llm, tools=tools, prompt=systemprompt)

#----------streamlit section--------------

# --- File Upload Handling ---
with st.sidebar:
    st.image('https://tse4.mm.bing.net/th/id/OIP.VxMvb7MAuyVVaWIcvpli3QHaE8?rs=1&pid=ImgDetMain&o=7&rm=3', width=100)
    st.title('EcoGrid Solutions')
    st.markdown('---')
    st.subheader('Upload your datas')

    file_prod = st.file_uploader("Solar production", type="csv", key="prod")
    file_cons = st.file_uploader("Domestic consumption", type="csv", key="cons")

    # --- File Upload Handling ---
    if file_prod and file_cons:
        try:
            # Save raw files to disk
            with open("solar_production_raw.csv", "wb") as f:
                f.write(file_prod.getbuffer())
            with open("household_consumption.csv", "wb") as f:
                f.write(file_cons.getbuffer())

            # Validate columns
            df_prod_check = pd.read_csv("solar_production_raw.csv")
            df_cons_check = pd.read_csv("household_consumption.csv")

            required_prod = {"timestamp", "kwh_produced"}
            required_cons = {"timestamp", "kwh_consumed"}

            if not required_prod.issubset(df_prod_check.columns):
                st.error(f"❌ Production file: missing columns. Expected: {required_prod}, found: {set(df_prod_check.columns)}")
                st.stop()

            if not required_cons.issubset(df_cons_check.columns):
                st.error(f"❌ Consumption file: missing columns. Expected: {required_cons}, found: {set(df_cons_check.columns)}")
                st.stop()

            os.makedirs("charts", exist_ok=True)

            if not st.session_state.uploaded_data:
                st.session_state.agent = agente()
                st.session_state.uploaded_data = True
                st.success("✅ Datas uploaded! EcoBot is ready.")

        except Exception as e:
            st.error(f"❌ Upload error: {str(e)}")
            st.stop()

        # ✅ Qui — dentro sidebar, dopo l'upload riuscito
        st.markdown("---")
        st.markdown("**💬 Example questions:**")
        st.markdown("- How much energy did I produce?")
        st.markdown("- Show me a chart")
        st.markdown("- How can I reduce my bill?")
        st.markdown("- Show me a comparison between produced and consumed energy")
        st.markdown("- How much energy am I expected to produce tomorrow?")

# ── Main chat interface ────────────────────────────────
st.title("🤖 EcoBot — Your energy analyst")
st.markdown("Ask me anything about your renewable energy!")
st.markdown("---")

# --- Chat Rendering ---
# Display all messages stored in the session (chat history)
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if "image" in msg:
            st.image(msg["image"])

# --- User Input Handling ---
# Input field for the user
if prompt := st.chat_input("Write your question..."):
    if not st.session_state.uploaded_data:
        # Ensure data has been uploaded before allowing interaction
        st.warning("Upload the CSV files from the sidebar first!")
    else:
        st.session_state.messages.append({"role": "user", "content": prompt})
        # Immediately display the user's message in the chat
        with st.chat_message("user"):
            st.markdown(prompt)
         # --- Assistant Response ---
        with st.chat_message("assistant"):
            with st.spinner("EcoBot is analyzing..."):
                result = st.session_state.agent.invoke({
                    "messages": [HumanMessage(content=prompt)]
                })
                answer = result["messages"][-1].content # Extract the latest response from the agent

            st.markdown(answer) # Display the assistant's response

            chart_path = "charts/analisi_energetica.png"
            # Keywords indicating a visualization request
            trigger_words = ["chart", "graph", "image", "plot", "visualize", # English
                             "grafico", "immagine", "visualizza", "confronto"] # Italian
            # If a chart exists and the user request suggests visualization → display image
            if os.path.exists(chart_path) and any(word in prompt.lower() for word in trigger_words):
                st.image(chart_path)
                # Save assistant message including the image
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer,
                    "image": chart_path
                })
            else:
                # Save only the text response
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer
                })

Writing app.py


In [ ]:
# Esegui questa cella per avviare il server

!streamlit run app.py

^C
